In [ ]:
# Lecture 25 — 5 Common Beginner Mistakes (and their fixes)
# Spark 101 · Module 7 · Lecture 25
#
# Import this file into a Databricks workspace as a notebook.
# Databricks recognizes `# COMMAND ----------` as a cell boundary on import.
# In VS Code / PyCharm this stays a plain Python file — same source, two homes.
#
# BIG-DATA DEMO: mistake #1 can OOM the Driver on purpose. Be ready to detach.
# Warn about DBU cost on screen and keep auto-termination on.

In [ ]:
# Partition counts, serverless-safe.
# df.rdd.getNumPartitions() uses the RDD API, which is BLOCKED on serverless
# compute. We count non-empty partitions via spark_partition_id() instead.
from pyspark.sql.functions import spark_partition_id

def part_count(df):
    return df.select(spark_partition_id()).distinct().count()

In [ ]:
print("Spark version:", spark.version)
# Databricks gives you `spark` for free — no SparkSession.builder needed.

In [ ]:
from pyspark.sql.functions import col, upper, when
big = spark.range(0, 50_000_000).withColumn("key", col("id") % 100)

In [ ]:
# --- MISTAKE 1: collect() on big data -> Driver OOM ------------------------
# DANGER: this pulls 50M rows to the Driver. Uncomment to demo the crash.
# rows = big.collect()          # <-- OOMs the Driver. This is the mistake.

# FIX: inspect with show()/take(); write the full result to storage.
big.show(5)                      # safe — a few rows
print(big.take(3))               # safe — a few rows
# big.write.mode("overwrite").parquet("file:/tmp/spark101_l25/out")  # full result

In [ ]:
# --- MISTAKE 2: wrong partition count -------------------------------------
print("partitions:", part_count(big))   # check FIRST
# FIX: repartition up for parallelism, coalesce down before writing.
print("repartitioned:", part_count(big.repartition(200)))
print("coalesced:", part_count(big.coalesce(8)))

In [ ]:
# --- MISTAKE 3: UDF overuse -----------------------------------------------
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

names = spark.createDataFrame([("ada",), ("linus",)], ["name"])

# The mistake: a UDF for something built-in already does.
slow = udf(lambda s: s.upper(), StringType())
names.withColumn("u", slow(col("name"))).show()      # BatchEvalPython in the plan

# FIX: use the built-in — Catalyst-friendly, no per-row serialization.
names.withColumn("u", upper(col("name"))).show()

In [ ]:
# --- MISTAKE 4: not caching reused data -----------------------------------
from pyspark.sql.functions import count
reused = big.filter(col("key") < 50).groupBy("key").agg(count("*").alias("n"))

# The mistake: three actions -> three full recomputations (see L23).
# FIX: cache what you reuse; unpersist when done.
reused.cache()
reused.count()          # computes + stores
reused.orderBy("n").take(5)   # reads cache
reused.unpersist()

In [ ]:
# --- MISTAKE 5: ignoring the Spark UI -------------------------------------
# There's no code fix — it's a habit. After running the cells above:
#   Spark UI -> Jobs / Stages / SQL
#   - task count per stage   (mistake 2: too few/many partitions)
#   - shuffle read/write      (mistake: needless shuffles, L22)
#   - BatchEvalPython node     (mistake 3: a UDF where a built-in would do)
#   - skew: one task far slower than the rest
# Open it BEFORE you change a line — every mistake above is visible there.

In [ ]:
# toPandas() is ALSO a full collect — same Driver-OOM risk as collect().
small_enough = big.limit(5).toPandas()   # fine ONLY because it's tiny
print(type(small_enough).__name__, "rows:", len(small_enough))
# Never call .toPandas() on a big DataFrame.

In [ ]:
# Column pruning before a wide op — read less, shuffle less.
from pyspark.sql.functions import count
big.select("key").groupBy("key").count().explain()   # ReadSchema = 1 column

In [ ]:
# Repeated actions WITHOUT cache recompute; cache fixes it (mistake #4 live).
reused = big.filter(col("key") < 10)
reused.cache(); reused.count(); reused.count()   # 2nd count reads the cache
reused.unpersist()

In [ ]:
# Recap:
#   1. collect() on big data -> Driver OOM. Use show/take/write.
#   2. Wrong partition count -> idle cores or overhead. repartition/coalesce.
#   3. UDF overuse -> slow. Search built-ins first.
#   4. No cache on reused data -> recompute. cache()/unpersist().
#   5. Ignoring the Spark UI -> guessing. Read it; the answer is there.
# End of Module 7 — Performance Basics.